# 0. Load Data and Set Up

In [96]:
# import core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# import machine learning libraries
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split

# set seaborn
sns.set_theme(context='notebook', style='darkgrid', palette='deep')
sns.color_palette()

[(0.2980392156862745, 0.4470588235294118, 0.6901960784313725),
 (0.8666666666666667, 0.5176470588235295, 0.3215686274509804),
 (0.3333333333333333, 0.6588235294117647, 0.40784313725490196),
 (0.7686274509803922, 0.3058823529411765, 0.3215686274509804),
 (0.5058823529411764, 0.4470588235294118, 0.7019607843137254),
 (0.5764705882352941, 0.47058823529411764, 0.3764705882352941),
 (0.8549019607843137, 0.5450980392156862, 0.7647058823529411),
 (0.5490196078431373, 0.5490196078431373, 0.5490196078431373),
 (0.8, 0.7254901960784313, 0.4549019607843137),
 (0.39215686274509803, 0.7098039215686275, 0.803921568627451)]

In [97]:
# Load Data

PATH = Path('/Users/alp/Projects/House Prices/data')

train = pd.read_csv(PATH / 'train.csv', index_col='Id')
test = pd.read_csv(PATH / 'test.csv', index_col='Id')

print("train.csv shape:", train.shape)
print("test.csv shape:", test.shape)


train.csv shape: (1460, 80)
test.csv shape: (1459, 79)


In [98]:
# Duplicate Detection
ids_uniq = len(set(train.index))
ids_total = train.shape[0]
ids_dupl = ids_total - ids_uniq

print(f"There are {ids_dupl} duplicated IDs for {ids_total} total entries")


There are 0 duplicated IDs for 1460 total entries


# 1. Preprocessing

## 1.1 Outliers

In [99]:
# Removing outliers detected in the EDA section
train = train.drop(train.SalePrice.nlargest(2).index)
train.shape

(1458, 80)

## 1.2 Null Values

In [100]:
# Detecting Features with Nulls
total_null = train.isnull().sum().sort_values(ascending=False)
percent_null = total_null / train.isnull().count().sort_values(ascending=False)
null_feats = pd.concat([total_null, percent_null], keys=['Total', 'Percent'], axis=1)
null_feats.head(20)

,Total,Percent
PoolQC,1452,0.995885
MiscFeature,1404,0.962963
Alley,1367,0.937586
Fence,1178,0.807956
MasVnrType,871,0.597394
FireplaceQu,690,0.473251
LotFrontage,259,0.177641
GarageYrBlt,81,0.055556
GarageCond,81,0.055556
GarageType,81,0.055556


In [101]:
train.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

In [102]:
# Dropping Features with more than 15% Null Rate
# train = train.drop(null_feats[(null_feats.Percent >= 0.15)].index, axis=1)    # Commented because next drop includes these
# Drop GarageX and BasemenX values with Null values since both have 
# features with no null rows that have higher correlation with target
train = train.drop((null_feats[null_feats.Total > 1]).index, axis=1)
# Drop MasVnrArea since it has low corr
# train = train.drop('MasVnrArea', axis=1)
# Drop the 1 row from Electrical that has a null value
train = train.drop((train[train.Electrical.isnull()]).index)

train.shape

(1457, 62)

In [103]:
# Do the same for test
test = test.drop((null_feats[null_feats.Total > 1]).index, axis=1)


In [104]:
(train.drop('SalePrice', axis=1).columns == test.columns).sum() == test.shape[1]

np.True_

## 1.3 Normalization

In [105]:
# log scale of target
train['SalePrice'] = np.log1p(train.SalePrice)
y = train.SalePrice

# 2. Model Training

## 2.1 Only highly correlated features

In [106]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import ElasticNet, Lasso, BayesianRidge, LassoLarsIC
import xgboost as xgb
import lightgbm as lgb
SEED = 42

In [107]:
# Only including highly correlated numeric features
corrmat = train.corr(numeric_only=True)

high_corr_feats = corrmat['SalePrice'].drop('SalePrice').abs().nlargest(17).index


In [108]:
X = train[high_corr_feats]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=.2)

In [109]:
def predict_error(X_train, X_val, y_train, y_val, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return root_mean_squared_error(y_val, y_pred)

In [110]:

models={
    'RandomForestRegressor': RandomForestRegressor(random_state=SEED),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=SEED),
    'ElasticNet': ElasticNet(random_state=SEED),
    'Lasso': Lasso(random_state=SEED),
    'LassoLarsIC': LassoLarsIC(),
}
errors = []

In [111]:
for model_name, model in models.items():
    err = predict_error(X_train, X_val, y_train, y_val, model)
    print(f'{model_name} score: {err}')
    errors.append(err)

RandomForestRegressor score: 0.17670942600277614
GradientBoostingRegressor score: 0.16954344085281184
ElasticNet score: 0.20972851664917236
Lasso score: 0.21660376307895313
LassoLarsIC score: 0.17885504827588614


In [112]:
# Null analysis for test
test[high_corr_feats].isna().sum().sort_values(ascending=False).head()


GarageCars     1
GarageArea     1
TotalBsmtSF    1
BsmtFinSF1     1
OverallQual    0
dtype: int64

In [113]:
test['GarageCars'] = test['GarageCars'].fillna(0.0)
test['GarageArea'] = test['GarageArea'].fillna(0.0)
test['TotalBsmtSF'] = test['TotalBsmtSF'].fillna(0.0)
test['BsmtFinSF1'] = test['BsmtFinSF1'].fillna(0.0)
test[high_corr_feats].isna().sum().sort_values(ascending=False).head()

OverallQual     0
TotRmsAbvGrd    0
2ndFlrSF        0
HalfBath        0
OpenPorchSF     0
dtype: int64

In [114]:
# Best one so far is GradientBoostingRegressor
# Create submission data for this model
model = GradientBoostingRegressor(random_state=SEED)
model.fit(X, y)
y_pred = model.predict(test[high_corr_feats])
# Don't forget to reverse the normalization 
y_pred = np.expm1(y_pred)
y_pred

array([124145.04786317, 150803.24330641, 187104.95931167, ...,
       170857.65359656, 114929.74115185, 255190.53145423], shape=(1459,))

In [131]:
temp = pd.read_csv(PATH / 'test.csv')

In [132]:
result = pd.concat([temp.Id, pd.Series(y_pred)], axis=1, keys=['Id', 'SalePrice'])


In [133]:
result

,Id,SalePrice
0,1461,124145.047863
1,1462,150803.243306
2,1463,187104.959312
3,1464,186500.744061
4,1465,192623.638712
...,...,...
1454,2915,68488.778404
1455,2916,83410.727642
1456,2917,170857.653597
1457,2918,114929.741152


In [134]:
result = result.set_index('Id')

In [135]:
result

,SalePrice
Id,
1461,124145.047863
1462,150803.243306
1463,187104.959312
1464,186500.744061
1465,192623.638712
...,...
2915,68488.778404
2916,83410.727642
2917,170857.653597


In [136]:
result.to_csv(PATH / 'submission1.csv' )